## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

from sklearn import preprocessing
from sklearn.decomposition import PCA

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

## Read

In [ ]:
# Read cleaned data

df = pd.read_csv('data/cleaned_credit_data2.csv')
print(df.shape)
df.head()

## Dummy Coding

In [ ]:
# One Hot Encoder - Creating Dummy Variables for 'Credit_Mix

ohe = preprocessing.OneHotEncoder(sparse=False)

oh_encoded = ohe.fit_transform(df[['Credit_Mix']])
df_oh_encoded = pd.DataFrame(oh_encoded, columns=ohe.get_feature_names_out(['Credit_Mix']))

df = pd.concat([df, df_oh_encoded], axis=1)

df.drop(['Credit_Mix'], axis=1, inplace=True)

df.head()


## Split Data (Test & Train)

In [ ]:
# Separate X from y

outcome = 'Credit_Score'
predictors = list(df.columns)
predictors.remove(outcome)

X = df[predictors]
y = df[outcome]

In [ ]:
# split dataset into training (70%) and test (30%) sets
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.3, random_state=42, stratify=y)

print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

## Label Encoding for y

In [ ]:
# 0 - Good, 1 - Poor, 2 - Standard

le = preprocessing.LabelEncoder()
y = le.fit_transform(y)

## Scaling

In [ ]:
# Standardize

scaler = preprocessing.StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## K-NN Classfication Model

In [ ]:
# Try out K-NN model with k=1

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
print('Accuracy: ', accuracy_score(y_test, y_pred))

In [ ]:
# Let's find the best number for k

results = []
for k in range(1, 100, 2):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_scaled, y_train)
    
    results.append({
        'k': k,
        'accuracy': accuracy_score(y_test, knn.predict(X_test_scaled))
    })

results_df = pd.DataFrame(results)
results_df.loc[results_df.accuracy == results_df.accuracy.max()]

In [ ]:
results_df.plot.scatter(x='k', y='accuracy', xlim=[0, 100]);

In [ ]:
# We choose k=29 based on the accuracy
# Confusion Matrix

knn = KNeighborsClassifier(n_neighbors=29)
knn.fit(X_train_scaled, y_train)

pred_y = knn.predict(X_test_scaled)
print(confusion_matrix(y_test, pred_y))
print('Accuracy:', accuracy_score(y_test, pred_y))

## Data Reduction

In [ ]:
# Let's find out the ideal number of components first

pcs = PCA()
pcs.fit(X_train_scaled)

# view the importance of principal components
pcsSummary_df = pd.DataFrame({
            'Proportion of variance': pcs.explained_variance_ratio_,
            'Cumulative proportion': np.cumsum(pcs.explained_variance_ratio_)
            })
pcsSummary_df.round(3)

In [ ]:
# Latent Root Criterion (Kaiser) and Scree Test

eig=pcs.explained_variance_.round(2)
print(pd.DataFrame(eig).T)

xvals = np.arange(X_train_scaled.shape[1]);

plt.figure(figsize=(10,5))
plt.plot(xvals, eig, 'ro-', linewidth=1)
plt.xticks(xvals)

plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue')

plt.axhline(y=1, color='b', linestyle='--')

1. Percentage of Variance Criterion: First 4

2. Latent Root Criterion / Kaiser Rule: First 3

3. Scree Test Criterion: First 2

In [ ]:
# 1. Percentage of Variance Criterion: First 4

pcs = PCA(n_components=4)
X_train_pcs = pcs.fit_transform(X_train_scaled)
X_test_pcs = pcs.transform(X_test_scaled)

results = []
for k in range(1, 200, 2):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_pcs, y_train)
    
    results.append({
        'k': k,
        'accuracy': accuracy_score(y_test, knn.predict(X_test_pcs))
    })

results_df = pd.DataFrame(results)
results_df.loc[results_df.accuracy == results_df.accuracy.max()]

In [ ]:
# 2. Latent Root Criterion / Kaiser Rule: First 3

pcs = PCA(n_components=3)
X_train_pcs = pcs.fit_transform(X_train_scaled)
X_test_pcs = pcs.transform(X_test_scaled)

results = []
for k in range(1, 100, 2):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_pcs, y_train)
    
    results.append({
        'k': k,
        'accuracy': accuracy_score(y_test, knn.predict(X_test_pcs))
    })

results_df = pd.DataFrame(results)
results_df.loc[results_df.accuracy == results_df.accuracy.max()]

In [ ]:
# 3. Scree Test Criterion: First 2

pcs = PCA(n_components=1)
X_train_pcs = pcs.fit_transform(X_train_scaled)
X_test_pcs = pcs.transform(X_test_scaled)

results = []
for k in range(1, 200, 2):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train_pcs, y_train)
    
    results.append({
        'k': k,
        'accuracy': accuracy_score(y_test, knn.predict(X_test_pcs))
    })

results_df = pd.DataFrame(results)
results_df.loc[results_df.accuracy == results_df.accuracy.max()]